# Content-Based Recommender: Similar Past Projects

This notebook implements the **content-based recommender** from `../06-recommender-systems-fundamentals.md`
-- one of the "multiple Recommender Systems" on the Virtual Liaison platform, suggesting similar past
projects (and, by the same mechanism, relevant cost-catalog items) given a new client request.

Fully offline: TF-IDF vectorization + cosine similarity over synthetic "project profiles", via
scikit-learn only.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
print("Ready.")

Ready.


## 1. Synthetic past-project profiles

Each past project is represented as a short text profile -- asset type, languages, region, and
therapeutic area -- the same kind of metadata the entity-extraction feature (Chapter 5) would capture
for a *new* request.

In [2]:
past_projects = [
    {"project_id": "atlas", "name": "Project Atlas",
     "profile": "Promotional deck localization Japanese Korean APAC region oncology launch"},
    {"project_id": "orion", "name": "Project Orion",
     "profile": "Regulatory submission package French German EU compliance review cardiology"},
    {"project_id": "nova", "name": "Project Nova",
     "profile": "Training deck localization Japanese APAC oncology launch medical education"},
    {"project_id": "vega", "name": "Project Vega",
     "profile": "Promotional campaign localization Spanish Portuguese LATAM region diabetes launch"},
    {"project_id": "sirius", "name": "Project Sirius",
     "profile": "Regulatory submission package German EU compliance review oncology safety update"},
    {"project_id": "lyra", "name": "Project Lyra",
     "profile": "Training deck localization Korean APAC region diabetes medical education"},
]

projects_df = pd.DataFrame(past_projects)
projects_df

,project_id,name,profile
0,atlas,Project Atlas,Promotional deck localization Japanese Korean APAC region oncology launch
1,orion,Project Orion,Regulatory submission package French German EU compliance review cardiology
2,nova,Project Nova,Training deck localization Japanese APAC oncology launch medical education
3,vega,Project Vega,Promotional campaign localization Spanish Portuguese LATAM region diabetes launch
4,sirius,Project Sirius,Regulatory submission package German EU compliance review oncology safety update
5,lyra,Project Lyra,Training deck localization Korean APAC region diabetes medical education


## 2. Vectorize project profiles with TF-IDF

Exactly the pattern from `../06-recommender-systems-fundamentals.md`: represent each project's text
profile as a TF-IDF vector, the same technique used for the "dense retrieval stand-in" throughout this
course.

In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
profile_matrix = vectorizer.fit_transform(projects_df["profile"])

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Profile matrix shape: {profile_matrix.shape}")

Vocabulary size: 28
Profile matrix shape: (6, 28)


## 3. Recommend top-N similar past projects for a new request

A new client request (already run through entity extraction in a real pipeline, per Chapter 5) is
vectorized with the *same* fitted vectorizer, and the most similar past projects are ranked by cosine
similarity -- the recommender re-uses the exact retrieval mechanics from Chapters 1-3, just applied to
project profiles instead of status/cost documents.

In [4]:
def recommend_similar_projects(new_request_text: str, top_n: int = 3) -> pd.DataFrame:
    new_vector = vectorizer.transform([new_request_text])
    similarities = cosine_similarity(new_vector, profile_matrix).ravel()

    results = projects_df.copy()
    results["similarity"] = similarities
    return results.sort_values("similarity", ascending=False).head(top_n).reset_index(drop=True)


new_request = "Localize promotional materials into Japanese for APAC oncology product launch"
recommendations = recommend_similar_projects(new_request, top_n=3)
recommendations

,project_id,name,profile,similarity
0,atlas,Project Atlas,Promotional deck localization Japanese Korean APAC region oncology launch,0.764139
1,nova,Project Nova,Training deck localization Japanese APAC oncology launch medical education,0.568095
2,vega,Project Vega,Promotional campaign localization Spanish Portuguese LATAM region diabetes launch,0.267623


The top recommendation should be **Project Atlas** (near-identical profile: promotional,
Japanese, APAC, oncology, launch), with **Project Nova** (Japanese, APAC, oncology, but "training
deck" not "promotional") a reasonable runner-up -- both clearly outrank the EU regulatory or LATAM
diabetes projects, which share almost no vocabulary with the new request.

## 4. A second example: cold-start resilience

Content-based recommendation works even for a **brand-new profile combination** never seen before in
the corpus (no interaction history required) -- the key advantage over collaborative filtering
discussed in Chapter 6.

In [5]:
novel_request = "Regulatory submission compliance package for a new diabetes therapy in the EU region"
novel_recommendations = recommend_similar_projects(novel_request, top_n=3)
novel_recommendations

,project_id,name,profile,similarity
0,orion,Project Orion,Regulatory submission package French German EU compliance review cardiology,0.611050
1,sirius,Project Sirius,Regulatory submission package German EU compliance review oncology safety update,0.590321
2,lyra,Project Lyra,Training deck localization Korean APAC region diabetes medical education,0.238829


`Project Sirius` and `Project Orion` (both regulatory/EU/compliance) should rank above the
localization-focused projects, even though this exact combination (diabetes + EU + regulatory)
never appeared as a single past project -- the recommender works from the *content* of the request,
not from having seen this exact request before.

## 5. Recommending cost-catalog items with the same mechanism

The same function works unchanged for the platform's second recommender use case -- suggesting
relevant cost-catalog items instead of past projects -- by simply swapping the corpus. This is the
concrete version of Chapter 6's point that "multiple Recommender Systems" on this platform likely
shared one underlying content-based/embedding-similarity mechanism rather than being built as
separate bespoke systems.

In [6]:
catalog_items = [
    {"sku": "SKU-LOC-JP-STD", "profile": "Japanese localization standard package promotional deck"},
    {"sku": "SKU-LOC-KR-STD", "profile": "Korean localization standard package training deck"},
    {"sku": "SKU-REG-EU-STD", "profile": "EU regulatory submission compliance review package"},
    {"sku": "SKU-LOC-ES-STD", "profile": "Spanish Portuguese localization standard package LATAM"},
]
catalog_df = pd.DataFrame(catalog_items)

catalog_vectorizer = TfidfVectorizer(stop_words="english")
catalog_matrix = catalog_vectorizer.fit_transform(catalog_df["profile"])


def recommend_catalog_items(request_text: str, top_n: int = 2) -> pd.DataFrame:
    vec = catalog_vectorizer.transform([request_text])
    sims = cosine_similarity(vec, catalog_matrix).ravel()
    results = catalog_df.copy()
    results["similarity"] = sims
    return results.sort_values("similarity", ascending=False).head(top_n).reset_index(drop=True)


recommend_catalog_items("Japanese localization for a new promotional deck", top_n=2)

,sku,profile,similarity
0,SKU-LOC-JP-STD,Japanese localization standard package promotional deck,0.903727
1,SKU-LOC-KR-STD,Korean localization standard package training deck,0.307011


## Takeaways

- Content-based filtering (TF-IDF + cosine similarity here; embeddings + Pinecone ANN search in
  production, per Chapter 6) recommends purely from item content, so it works immediately for new
  projects and new request combinations -- no interaction history required, unlike collaborative
  filtering.
- The exact same `recommend_*` pattern serves both "similar past projects" and "relevant cost-catalog
  items" -- one mechanism, two applications, which is why a production version of this platform likely
  didn't need a separate bespoke recommender per feature.
- See `../06-recommender-systems-fundamentals.md` for how this generalizes to real embeddings and
  Pinecone, and for the discussion of when collaborative-filtering signals would be worth layering in
  as the platform accumulates interaction history.